# AIC26 GLAP audio↔text encoder (standalone Kaggle server)

Chạy RIÊNG khỏi notebook PE để tránh xung đột `transformers` (GLAP cần <5.0, PE có thể dùng bản khác).
Phục vụ `/encode-audio-text` → vector **1024-d** khớp model đã tạo `.glap.npy` (`mispeech/GLAP`, collection `aic26_audio_glap_v1`).

**Cách dùng:** bật Internet (GPU không bắt buộc — text encoder chạy CPU), chạy tất cả cell → lấy `PUBLIC_URL` → đặt `GLAP_ENCODER_URL=<PUBLIC_URL>` trong `backend/.env`.

In [ ]:
# GLAP cần transformers < 5.0 (nếu không: ImportError GenerationMixin from transformers.generation).
!pip install glap_model || pip install "git+https://github.com/xiaomi-research/dasheng-glap.git"
!pip install "transformers==4.53.3"
!pip install -U fastapi uvicorn nest_asyncio pydantic
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared
!python -c "import transformers; from glap_model import glap_inference; print('transformers', transformers.__version__, '| glap_model OK')"

In [ ]:
import os, time, threading
from typing import List, Optional
import numpy as np, torch
from glap_model import glap_inference

# CPU: encode text query rất nhẹ; tránh lỗi 'index is on cpu, different from cuda:0'
# (glap.encode_text không tự đưa token ids lên GPU). Embedding vẫn cùng không gian .glap.npy.
DEVICE = "cpu"
print("Loading GLAP (mispeech/GLAP) on", DEVICE)
glap_model = glap_inference()
try:
    glap_model = glap_model.to(DEVICE).eval()
except Exception:
    pass
glap_lock = threading.Lock()

@torch.inference_mode()
def encode_glap_text(texts: List[str]) -> np.ndarray:
    if isinstance(texts, str):
        texts = [texts]
    emb = glap_model.encode_text(list(texts)).float()
    emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)  # L2-norm = giống .glap.npy
    return emb.detach().cpu().numpy().astype("float32")

_v = encode_glap_text(["a car horn honking", "piano music"])
print("GLAP text dim:", _v.shape, "| norm:", float(np.linalg.norm(_v[0])))

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
import nest_asyncio, uvicorn

SERVICE_TOKEN = os.environ.get("AIC26_SERVICE_TOKEN", "change-me-dev-token")
app = FastAPI(title="AIC26 GLAP audio-text encoder")

def check_auth(authorization: Optional[str]):
    if SERVICE_TOKEN == "change-me-dev-token":
        return
    if authorization != f"Bearer {SERVICE_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized")

class EncodeAudioTextRequest(BaseModel):
    texts: List[str] = Field(..., min_length=1)

@app.get("/health")
def health():
    return {"ok": True, "model": "mispeech/GLAP", "device": DEVICE, "dim": 1024}

@app.post("/encode-audio-text")
def api_encode_audio_text(req: EncodeAudioTextRequest, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    t0 = time.time()
    with glap_lock:
        vectors = encode_glap_text(req.texts)
    return {"model": "mispeech/GLAP", "dim": int(vectors.shape[1]),
            "latency_ms": round((time.time() - t0) * 1000, 2), "vectors": vectors.tolist()}

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"), daemon=True).start()
print("GLAP server on http://localhost:8000")

In [ ]:
import subprocess, re, time
!pkill -f cloudflared || true
proc = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
for _ in range(80):
    line = proc.stdout.readline(); print(line, end="")
    m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if m: PUBLIC_URL = m.group(0); break
    time.sleep(0.5)
print("\nGLAP_ENCODER_URL =", PUBLIC_URL)

In [ ]:
import requests
r = requests.post("http://127.0.0.1:8000/encode-audio-text", json={"texts": ["piano music", "tiếng còi xe"]}, timeout=60)
j = r.json(); print("status", r.status_code, "| dim", j["dim"], "| vec_len", len(j["vectors"][0]))